# 01 — Data exploration

Descriptive statistics for both corpora, feeding D1 section 6.

Run `make data` first for the real corpora, or `python -m frd.make_synthetic`
for the offline stand-in. Anything computed from `synthetic_*` is a smoke
test and belongs in no report.

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
from frd import datasets

datasets.available()

## Load

Swap the keys for `ott` and `salminen` once the real corpora are in place.

In [ ]:
HUMAN, MACHINE = "synthetic_human", "synthetic_machine"
human, machine = datasets.load_pair(HUMAN, MACHINE)
human.head()

## Descriptive statistics

In [ ]:
pd.DataFrame({
    "human-written fakes": datasets.describe(human),
    "machine-generated fakes": datasets.describe(machine),
})

## Review length

Length is the first place the two populations differ. Worth checking
before assuming a model has learned anything about deception.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, frame, title in zip(axes, (human, machine), ("Human-written", "Machine-generated")):
    for label, name in ((0, "genuine"), (1, "fake")):
        subset = frame[frame["label"] == label]["text"].str.split().str.len()
        ax.hist(subset, bins=30, alpha=0.6, label=name)
    ax.set_title(title); ax.set_xlabel("words"); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("reviews"); axes[0].legend(frameon=False)
plt.tight_layout()

## Overlap between the corpora

If the two corpora shared vocabulary heavily, the cross-dataset drop
would be harder to attribute to the fake-review population rather than
to the domain. Worth measuring, and worth stating in the limitations.

In [ ]:
def vocabulary(frame):
    return {token.lower().strip(".,!?") for text in frame["text"] for token in text.split()}

a, b = vocabulary(human), vocabulary(machine)
print(f"human {len(a)}  machine {len(b)}  shared {len(a & b)}")
print(f"Jaccard {len(a & b) / len(a | b):.3f}")